# 01 — PaySim preprocessing (simple)


In [15]:
from pathlib import Path
import json
import os

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "raw" / "paysim1.csv"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

PARQUET_OUT = PROCESSED / "transactions_clean.parquet"
JSON_OUT = PROCESSED / "monthly_statements.json"

print("ROOT:", ROOT)
print("RAW:", RAW_PATH, "exists:", RAW_PATH.is_file())

ROOT: /mnt/data-disk/FinSight/FinSight-AI
RAW: /mnt/data-disk/FinSight/FinSight-AI/data/raw/paysim1.csv exists: True


In [16]:
with open(RAW_PATH, "rb") as f:
    n_lines = sum(buf.count(b"\n") for buf in iter(lambda: f.read(8 * 1024 * 1024), b""))
row_count = n_lines - 1  # minus header
print(f"Number of Rows: {row_count:,}")


Number of Rows: 6,362,620


In [17]:
MAX_ROWS = 500_000
CHUNKSIZE = 50_000

if RAW_PATH.is_file():
    print("File exists: Loading started")

if MAX_ROWS is not None:
    df = pd.read_csv(RAW_PATH, nrows=int(MAX_ROWS))
else:
    parts = []
    for chunk in tqdm(
        pd.read_csv(RAW_PATH, chunksize=CHUNKSIZE),
        desc="read_csv chunks",
    ):
        parts.append(chunk)
    df = pd.concat(parts, ignore_index=True)

print(df.shape)
df.head()

File exists: Loading started
(500000, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [32]:
null_counts = df.isnull().sum()
print(null_counts)


step          0
type          0
amount        0
nameOrig      0
nameDest      0
isFraud       0
timestamp     0
month         0
date          0
category      0
is_anomaly    0
dtype: int64


In [19]:
print(df.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            500000 non-null  int64  
 1   type            500000 non-null  object 
 2   amount          500000 non-null  float64
 3   nameOrig        500000 non-null  object 
 4   oldbalanceOrg   500000 non-null  float64
 5   newbalanceOrig  500000 non-null  float64
 6   nameDest        500000 non-null  object 
 7   oldbalanceDest  500000 non-null  float64
 8   newbalanceDest  500000 non-null  float64
 9   isFraud         500000 non-null  int64  
 10  isFlaggedFraud  500000 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 42.0+ MB


In [21]:
df.sample(5)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
414496,18,CASH_OUT,269171.93,C872111598,0.00,0.00,C355970563,4146695.87,3949629.61,0,0
95787,10,CASH_OUT,128954.50,C259847082,0.00,0.00,C35617324,179293.15,1528613.60,0,0
254263,14,CASH_OUT,87732.88,C615148076,0.00,0.00,C1740277954,1471493.83,2225537.83,0,0
333119,16,PAYMENT,31592.30,C1054205440,252583.11,220990.81,M1809692181,0.00,0.00,0,0
349764,17,CASH_OUT,576402.53,C1433196975,0.00,0.00,C8290162,10759801.45,11644683.84,0,0


In [22]:
cols = ["step", "type", "amount", "nameOrig", "nameDest", "isFraud"]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[cols].copy()
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df = df.dropna(subset=["amount"])
df = df[df["amount"] > 0]
df.shape

(500000, 6)

In [23]:
base = pd.Timestamp("2024-01-01")
df["timestamp"] = base + pd.to_timedelta(df["step"], unit="h")
df["month"] = df["timestamp"].dt.strftime("%Y-%m")
df["date"] = df["timestamp"].dt.date.astype(str)

In [24]:
category_map = {
    "PAYMENT": "Bills & Purchases",
    "TRANSFER": "Transfer",
    "CASH_OUT": "Withdrawal",
    "CASH_IN": "Deposit",
    "DEBIT": "Debit",
}
df["category"] = df["type"].map(category_map).fillna("Other")

In [25]:
threshold = df["amount"].mean() + 3 * df["amount"].std()
df["is_anomaly"] = (df["isFraud"] == 1) | (df["amount"] > threshold)
df["is_anomaly"].value_counts()

is_anomaly
False    490041
True       9959
Name: count, dtype: int64

In [26]:
df.head()

,step,type,amount,nameOrig,nameDest,isFraud,timestamp,month,date,category,is_anomaly
0,1,PAYMENT,9839.64,C1231006815,M1979787155,0,2024-01-01 01:00:00,2024-01,2024-01-01,Bills & Purchases,False
1,1,PAYMENT,1864.28,C1666544295,M2044282225,0,2024-01-01 01:00:00,2024-01,2024-01-01,Bills & Purchases,False
2,1,TRANSFER,181.00,C1305486145,C553264065,1,2024-01-01 01:00:00,2024-01,2024-01-01,Transfer,True
3,1,CASH_OUT,181.00,C840083671,C38997010,1,2024-01-01 01:00:00,2024-01,2024-01-01,Withdrawal,True
4,1,PAYMENT,11668.14,C2048537720,M1230701703,0,2024-01-01 01:00:00,2024-01,2024-01-01,Bills & Purchases,False


In [27]:

def month_statement(sub: pd.DataFrame, month: str) -> dict:
    tot = float(sub["amount"].sum())
    dep = sub.loc[sub["category"] == "Deposit", "amount"]
    total_received = float(dep.sum())
    total_spent = float(sub.loc[sub["category"] != "Deposit", "amount"].sum())
    by_cat = (
        sub.groupby("category", dropna=False)["amount"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "total", "count": "count"})
    )
    by_cat["pct"] = (by_cat["total"] / tot * 100).round(4) if tot else 0.0
    cat_rows = [
        {
            "name": name,
            "total": float(row["total"]),
            "count": int(row["count"]),
            "percentage": float(row["pct"]),
        }
        for name, row in by_cat.iterrows()
    ]
    return {
        "month": month,
        "transaction_count": int(len(sub)),
        "amount_total": tot,
        "total_received": total_received,
        "total_spent_non_deposit": total_spent,
        "fraud_count": int((sub["isFraud"] == 1).sum()),
        "anomaly_count": int(sub["is_anomaly"].sum()),
        "by_category": sorted(cat_rows, key=lambda x: x["total"], reverse=True),
    }


monthly_statements = {}
for m, g in df.groupby("month"):
    monthly_statements[m] = month_statement(g, m)

list(monthly_statements.keys())[:5], len(monthly_statements)

(['2024-01'], 1)

In [28]:
df.to_parquet(PARQUET_OUT, index=False)
with open(JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(monthly_statements, f, indent=2, ensure_ascii=False)

print("Wrote:", PARQUET_OUT)
print("Wrote:", JSON_OUT)

Wrote: /mnt/data-disk/FinSight/FinSight-AI/data/processed/transactions_clean.parquet
Wrote: /mnt/data-disk/FinSight/FinSight-AI/data/processed/monthly_statements.json
